In [15]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from io import StringIO

# Direct scrape approach
url = "https://en.wikipedia.org/wiki/U.S._Open_(golf)"

headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# Fetch with proper headers first
response = requests.get(url, headers=headers)
print(f"Status: {response.status_code}")

# Parse tables from the HTML
dfs = pd.read_html(StringIO(response.text))
print(f"Found {len(dfs)} tables")

Status: 200
Found 16 tables


In [16]:
# Preview each table to find the Winners table
for i, df in enumerate(dfs[:12]):
    print(f"\n=== Table {i} === Shape: {df.shape}")
    print(df.columns.tolist()[:5])  # First 5 columns
    print(df.head(2))


=== Table 0 === Shape: (13, 2)
[0, 1]
                        0                       1
0                     NaN                     NaN
1  Tournament information  Tournament information

=== Table 1 === Shape: (1, 2)
[0, 1]
    0                                                  1
0 NaN  This section needs additional citations for ve...

=== Table 2 === Shape: (127, 9)
['Year', 'Winner', 'Score', 'To par', 'Margin of victory']
   Year                 Winner Score To par Margin of victory  \
0  2025            J. J. Spaun   279     –1         2 strokes   
1  2024  Bryson DeChambeau (2)   274     −6          1 stroke   

       Runner(s)-up Winner's share ($)                            Venue  \
0  Robert MacIntyre            4300000                          Oakmont   
1      Rory McIlroy            4300000  Pinehurst Resort (Course No. 2)   

                    Location  
0         Plum, Pennsylvania  
1  Pinehurst, North Carolina  

=== Table 3 === Shape: (4, 1)
['Legend']
          

In [ ]:
# Scrape the venue links from the Winners table HTML
# (using the same response from Cell 0)
soup = BeautifulSoup(response.text, 'html.parser')

# Find the Winners table (it has Year, Winner, Venue columns)
tables = soup.find_all('table', class_='wikitable')
print(f"Found {len(tables)} wikitables")

venue_links = []

for table in tables:
    # Check if this table has a Venue header
    headers_row = table.find('tr')
    if headers_row:
        ths = headers_row.find_all('th')
        header_texts = [th.get_text(strip=True).lower() for th in ths]
        
        if 'venue' in header_texts:
            print(f"Found Winners table with headers: {header_texts}")
            venue_idx = header_texts.index('venue')
            
            # Extract venue links from each row
            for row in table.find_all('tr')[1:]:  # Skip header
                cells = row.find_all(['td', 'th'])
                if len(cells) > venue_idx:
                    venue_cell = cells[venue_idx]
                    link = venue_cell.find('a')
                    if link and link.get('href', '').startswith('/wiki/'):
                        venue_name = link.get('title', link.get_text(strip=True))
                        venue_href = link.get('href')
                        year_cell = cells[0] if cells else None
                        year = year_cell.get_text(strip=True) if year_cell else ''
                        
                        venue_links.append({
                            'year': year,
                            'venue': venue_name,
                            'wiki_path': venue_href
                        })
            break

print(f"\nExtracted {len(venue_links)} venue links")
venue_links[:10]

Found 4 wikitables
Found Winners table with headers: ['year', 'winner', 'score', 'to par', 'margin ofvictory', 'runner(s)-up', "winner'sshare ($)", 'venue', 'location']

Extracted 125 venue links


[{'year': '2025',
  'venue': 'Oakmont Country Club',
  'wiki_path': '/wiki/Oakmont_Country_Club'},
 {'year': '2024',
  'venue': 'Pinehurst Resort',
  'wiki_path': '/wiki/Pinehurst_Resort'},
 {'year': '2023',
  'venue': 'Los Angeles Country Club',
  'wiki_path': '/wiki/Los_Angeles_Country_Club'},
 {'year': '2022',
  'venue': 'The Country Club',
  'wiki_path': '/wiki/The_Country_Club'},
 {'year': '2021',
  'venue': 'Torrey Pines Golf Course',
  'wiki_path': '/wiki/Torrey_Pines_Golf_Course'},
 {'year': '2020',
  'venue': 'Winged Foot Golf Club',
  'wiki_path': '/wiki/Winged_Foot_Golf_Club'},
 {'year': '2019',
  'venue': 'Pebble Beach Golf Links',
  'wiki_path': '/wiki/Pebble_Beach_Golf_Links'},
 {'year': '2018',
  'venue': 'Shinnecock Hills Golf Club',
  'wiki_path': '/wiki/Shinnecock_Hills_Golf_Club'},
 {'year': '2017', 'venue': 'Erin Hills', 'wiki_path': '/wiki/Erin_Hills'},
 {'year': '2016',
  'venue': 'Oakmont Country Club',
  'wiki_path': '/wiki/Oakmont_Country_Club'}]

In [ ]:
# Get unique venues
unique_venues = {}
for v in venue_links:
    venue = v['venue']
    if venue not in unique_venues:
        unique_venues[venue] = v['wiki_path']

print(f"Found {len(unique_venues)} unique U.S. Open venues:\n")
for venue, path in list(unique_venues.items())[:30]:
    print(f"  - {venue}: {path}")

Found 53 unique U.S. Open venues:

  - Oakmont Country Club: /wiki/Oakmont_Country_Club
  - Pinehurst Resort: /wiki/Pinehurst_Resort
  - Los Angeles Country Club: /wiki/Los_Angeles_Country_Club
  - The Country Club: /wiki/The_Country_Club
  - Torrey Pines Golf Course: /wiki/Torrey_Pines_Golf_Course
  - Winged Foot Golf Club: /wiki/Winged_Foot_Golf_Club
  - Pebble Beach Golf Links: /wiki/Pebble_Beach_Golf_Links
  - Shinnecock Hills Golf Club: /wiki/Shinnecock_Hills_Golf_Club
  - Erin Hills: /wiki/Erin_Hills
  - Chambers Bay: /wiki/Chambers_Bay
  - Merion Golf Club: /wiki/Merion_Golf_Club
  - Olympic Club: /wiki/Olympic_Club
  - Congressional Country Club: /wiki/Congressional_Country_Club
  - Bethpage State Park: /wiki/Bethpage_State_Park
  - Olympia Fields Country Club: /wiki/Olympia_Fields_Country_Club
  - Southern Hills Country Club: /wiki/Southern_Hills_Country_Club
  - Oakland Hills Country Club: /wiki/Oakland_Hills_Country_Club
  - Baltusrol Golf Club: /wiki/Baltusrol_Golf_Club
  -

In [ ]:
# Function to get coordinates from a course's Wikipedia page
import re

def dms_to_decimal(degrees, minutes, seconds, direction):
    """Convert DMS (Degrees, Minutes, Seconds) to decimal degrees"""
    decimal = float(degrees) + float(minutes)/60 + float(seconds)/3600
    if direction in ['S', 'W']:
        decimal = -decimal
    return decimal

def parse_dms_coords(text):
    """Parse coordinates like '36°34′07″N 121°57′02″W' to decimal"""
    # Pattern for DMS: 36°34′07″N or 36°34'07"N (different quote styles)
    pattern = r"(\d+)°(\d+)[′'](\d+)[″\"]([NSEW])"
    matches = re.findall(pattern, text)
    
    if len(matches) >= 2:
        lat_match = matches[0]
        lng_match = matches[1]
        
        lat = dms_to_decimal(lat_match[0], lat_match[1], lat_match[2], lat_match[3])
        lng = dms_to_decimal(lng_match[0], lng_match[1], lng_match[2], lng_match[3])
        
        return {'lat': lat, 'lng': lng}
    return None

def get_coords_from_wiki_path(wiki_path):
    """Scrape coordinates from a Wikipedia page using the wiki path"""
    wiki_url = f"https://en.wikipedia.org{wiki_path}"
    try:
        resp = requests.get(wiki_url, headers=headers, timeout=10)
        if resp.status_code != 200:
            return None
        
        page_soup = BeautifulSoup(resp.text, 'html.parser')
        
        # Method 1: Look for the coordinates row in the infobox
        # Find span with class "geo-dms" or similar
        geo_dms = page_soup.find('span', class_='geo-dms')
        if geo_dms:
            coords = parse_dms_coords(geo_dms.get_text())
            if coords:
                return coords
        
        # Method 2: Try to find geo coordinates span (format: "36.5681; -121.9486")
        geo = page_soup.find('span', class_='geo')
        if geo:
            coords_text = geo.get_text()
            parts = coords_text.replace(';', ' ').split()
            if len(parts) >= 2:
                return {
                    'lat': float(parts[0]),
                    'lng': float(parts[1])
                }
        
        # Method 3: Look for coordinates anywhere in the infobox
        infobox = page_soup.find('table', class_='infobox')
        if infobox:
            infobox_text = infobox.get_text()
            coords = parse_dms_coords(infobox_text)
            if coords:
                return coords
                
    except Exception as e:
        print(f"Error: {e}")
    return None

# Test with a few known courses
test_courses = [
    ("/wiki/Pebble_Beach_Golf_Links", "Pebble Beach", 36.57, -121.95),
    ("/wiki/Augusta_National_Golf_Club", "Augusta National", 33.50, -82.02),
    ("/wiki/Oakmont_Country_Club", "Oakmont", 40.53, -79.83),
]

print("Verifying coordinates (DMS → Decimal conversion):")
for path, name, expected_lat, expected_lng in test_courses:
    coords = get_coords_from_wiki_path(path)
    if coords:
        lat_diff = abs(coords['lat'] - expected_lat)
        lng_diff = abs(coords['lng'] - expected_lng)
        status = "✓" if lat_diff < 0.1 and lng_diff < 0.1 else "⚠️ OFF"
        print(f"  {status} {name}: {coords['lat']:.5f}, {coords['lng']:.5f} (expected ~{expected_lat}, {expected_lng})")
    else:
        print(f"  ✗ {name}: No coords")

Verifying coordinates (DMS → Decimal conversion):
  ✓ Pebble Beach: 36.56861, -121.95056 (expected ~36.57, -121.95)
  ✓ Augusta National: 33.50250, -82.02000 (expected ~33.5, -82.02)
  ✓ Oakmont: 40.52611, -79.82639 (expected ~40.53, -79.83)


In [ ]:
# Get coordinates for all unique venues
import time

courses_data = []

for venue_name, wiki_path in unique_venues.items():
    coords = get_coords_from_wiki_path(wiki_path)
    if coords:
        courses_data.append({
            'name': venue_name,
            'wiki_path': wiki_path,
            'lat': coords['lat'],
            'lng': coords['lng']
        })
        print(f"✓ {venue_name}: {coords['lat']}, {coords['lng']}")
    else:
        print(f"✗ {venue_name}: No coords")
    time.sleep(0.2)  # Be nice to Wikipedia

print(f"\n✅ Got coordinates for {len(courses_data)}/{len(unique_venues)} venues")

✓ Oakmont Country Club: 40.52611111111111, -79.82638888888889
✓ Pinehurst Resort: 35.18944444444444, -79.46777777777778
✓ Los Angeles Country Club: 34.06888888888889, -118.42305555555556
✓ The Country Club: 42.31333333333333, -71.15083333333334
✓ Torrey Pines Golf Course: 32.904444444444444, -117.24527777777777
✓ Winged Foot Golf Club: 40.962500000000006, -73.75361111111111
✓ Pebble Beach Golf Links: 36.56861111111112, -121.95055555555555
✓ Shinnecock Hills Golf Club: 40.89388888888889, -72.44
✓ Erin Hills: 43.245, -88.39500000000001
✓ Chambers Bay: 47.2, -122.57
✓ Merion Golf Club: 40.00111111111111, -75.31194444444444
✓ Olympic Club: 37.70888888888889, -122.495
✓ Congressional Country Club: 38.99611111111111, -77.17694444444444
✓ Bethpage State Park: 40.7525, -73.4675
✓ Olympia Fields Country Club: 41.52111111111111, -87.68694444444445
✓ Southern Hills Country Club: 36.0726361, -95.9496583
✓ Oakland Hills Country Club: 42.54388888888889, -83.27694444444444
✓ Baltusrol Golf Club: 40.7

In [ ]:
# Get years and winners data from the Winners table (Table 2)
winners_df = dfs[2]
print(f"Winners table shape: {winners_df.shape}")
print(f"Columns: {winners_df.columns.tolist()}")

# Clean the data - remove cancelled years
winners_df = winners_df[~winners_df['Year'].astype(str).str.contains('Cancelled|World War', na=False)]
winners_df = winners_df[pd.to_numeric(winners_df['Year'], errors='coerce').notna()]

# Build a mapping of venue -> {years, winners, location}
venue_data = {}
for _, row in winners_df.iterrows():
    venue = str(row['Venue']).strip()
    year = str(row['Year'])
    winner = str(row['Winner']).strip()
    location = str(row.get('Location', '')).strip()
    
    # Clean winner name (remove numbers in parentheses like "(2)")
    winner_clean = re.sub(r'\s*\(\d+\)\s*$', '', winner)
    
    if venue not in venue_data:
        venue_data[venue] = {'years': [], 'winners': [], 'location': location}
    
    venue_data[venue]['years'].append(year)
    venue_data[venue]['winners'].append(winner_clean)

print(f"\nBuilt data for {len(venue_data)} venues")

# Show sample
for venue, data in list(venue_data.items())[:5]:
    print(f"  {venue}: {data['years'][:3]}... | Winners: {data['winners'][:2]}...")

Winners table shape: (127, 9)
Columns: ['Year', 'Winner', 'Score', 'To par', 'Margin of victory', 'Runner(s)-up', "Winner's share ($)", 'Venue', 'Location']

Built data for 57 venues
  Oakmont: ['2025', '2016', '2007']... | Winners: ['J. J. Spaun', 'Dustin Johnson']...
  Pinehurst Resort (Course No. 2): ['2024', '2014', '2005']... | Winners: ['Bryson DeChambeau', 'Martin Kaymer']...
  Los Angeles Country Club (North Course): ['2023']... | Winners: ['Wyndham Clark']...
  The Country Club (Composite Course): ['2022', '1988', '1963']... | Winners: ['Matt Fitzpatrick', 'Curtis Strange']...
  Torrey Pines (South Course): ['2021', '2008']... | Winners: ['Jon Rahm', 'Tiger Woods']...


In [ ]:
# Export to JSON with full data
import json

def match_venue_name(course_name, venue_data):
    """Try to match course name to venue data"""
    course_lower = course_name.lower()
    
    for venue, data in venue_data.items():
        venue_lower = venue.lower()
        # Direct match or partial match
        if venue_lower in course_lower or course_lower in venue_lower:
            return data
        # Match key words
        venue_words = set(venue_lower.replace('(', ' ').replace(')', ' ').split())
        course_words = set(course_lower.replace('(', ' ').replace(')', ' ').split())
        common = venue_words & course_words - {'golf', 'club', 'country', 'the', 'course', 'no.', '2'}
        if len(common) >= 1:
            return data
    return None

# Build final output
courses_df = pd.DataFrame(courses_data)
print(f"Courses with coordinates: {len(courses_df)}")

output = {"courses": []}
matched = 0

for _, row in courses_df.iterrows():
    course_id = row['name'].lower().replace(' ', '_')
    for char in ['(', ')', ',', "'", '"']:
        course_id = course_id.replace(char, '')
    
    # Try to match with venue data
    vdata = match_venue_name(row['name'], venue_data)
    
    # Parse city/state from location if available
    city, state = "", ""
    if vdata and vdata['location']:
        parts = vdata['location'].split(',')
        if len(parts) >= 2:
            city = parts[0].strip()
            state = parts[-1].strip()
    
    course_entry = {
        "id": course_id,
        "name": row['name'],
        "city": city,
        "state": state,
        "country": "USA",
        "coordinates": {
            "lat": row['lat'],
            "lng": row['lng']
        },
        "tournaments": ["U.S. Open"],
        "us_open_years": vdata['years'] if vdata else [],
        "recent_winners": vdata['winners'][:5] if vdata else [],  # Last 5 winners
        "recent_winner": vdata['winners'][0] if vdata and vdata['winners'] else ""
    }
    
    if vdata:
        matched += 1
    
    output["courses"].append(course_entry)

print(f"Matched {matched}/{len(courses_df)} courses with year/winner data")

# Save to file
with open('../quizzes/startingtee/us_open_courses.json', 'w') as f:
    json.dump(output, f, indent=2)
    
print(f"\n✅ Saved {len(output['courses'])} courses to us_open_courses.json")

Courses with coordinates: 52
Matched 51/52 courses with year/winner data

✅ Saved 52 courses to us_open_courses.json


In [ ]:
# Preview sample courses
print("Sample courses with full data:\n")
for course in output['courses'][:5]:
    print(f"📍 {course['name']}")
    print(f"   Location: {course['city']}, {course['state']}")
    print(f"   Coords: ({course['coordinates']['lat']:.4f}, {course['coordinates']['lng']:.4f})")
    print(f"   U.S. Open years: {course['us_open_years'][:3]}...")
    print(f"   Recent winners: {course['recent_winners'][:2]}")
    print()

Sample courses with full data:

📍 Oakmont Country Club
   Location: Plum, Pennsylvania
   Coords: (40.5261, -79.8264)
   U.S. Open years: ['2025', '2016', '2007']...
   Recent winners: ['J. J. Spaun', 'Dustin Johnson']

📍 Pinehurst Resort
   Location: Pinehurst, North Carolina
   Coords: (35.1894, -79.4678)
   U.S. Open years: ['2024', '2014', '2005']...
   Recent winners: ['Bryson DeChambeau', 'Martin Kaymer']

📍 Los Angeles Country Club
   Location: Los Angeles, California
   Coords: (34.0689, -118.4231)
   U.S. Open years: ['2023']...
   Recent winners: ['Wyndham Clark']

📍 The Country Club
   Location: Brookline, Massachusetts
   Coords: (42.3133, -71.1508)
   U.S. Open years: ['2022', '1988', '1963']...
   Recent winners: ['Matt Fitzpatrick', 'Curtis Strange']

📍 Torrey Pines Golf Course
   Location: San Diego, California
   Coords: (32.9044, -117.2453)
   U.S. Open years: ['2021', '2008']...
   Recent winners: ['Jon Rahm', 'Tiger Woods']

